# 개미의 우산 (Ant's Umbrella) - Google Colab 통합 ML 파이프라인 구동 가이드

본 노트북은 로컬 PC 자원의 한계로 인해 발생하는 크래시(재부팅, 세션 종료)를 방지하고, Google Colab에서 제공하는 **무료 GPU 가속(T4 GPU 등)**을 사용하여 HuggingFace 모델 연산(FinBERT, Zero-shot 등) 및 XGBoost 학습 파이프라인을 안정적이고 빠르게 구동하기 위한 가이드라인입니다.

### 💡 런타임 유형 설정
반드시 상단 메뉴의 **런타임** -> **런타임 유형 변경**에서 **T4 GPU** (혹은 사용 가능한 GPU)를 선택해 주세요. GPU 가속을 활성화하지 않으면 연산 속도가 매우 느려지며 제한이 걸릴 수 있습니다.

In [ ]:
# GPU 가속 상태 확인
import torch
print("==========================================")
print("GPU 가속 상태 확인")
print("==========================================")
print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"사용 중인 GPU 디바이스: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU 가속이 활성화되지 않았습니다.")
    print("상단 메뉴 [런타임] > [런타임 유형 변경]에서 T4 GPU를 선택해 주세요.")

### 1단계. Google Drive 마운트 또는 GitHub Repository 복제

코랩 인스턴스에 코드를 가져오는 두 가지 방법 중 하나를 선택해 주세요.
* **방법 A**: 구글 드라이브에 프로젝트 전체를 업로드한 경우 (드라이브 마운트)
* **방법 B**: GitHub 저장소를 활용해 바로 Clone하여 작업할 경우

In [ ]:
# 방법 A: Google Drive 마운트 (구글 드라이브에 프로젝트 폴더가 있는 경우)
# 필요에 따라 주석을 해제하고 실행하세요.
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/ants-umbrella

In [ ]:
# 방법 B: GitHub 레포지토리 직접 클론
# 실제 본인의 깃허브 저장소 주소로 변경하여 실행하세요.
# !git clone https://github.com/YOUR_GITHUB_REPO/ants-umbrella.git
# %cd ants-umbrella

### 2단계. 필수 라이브러리(Dependencies) 설치

프로젝트의 `pyproject.toml`에 정의된 핵심 의존성 패키지들을 Colab 파이썬 환경에 설치합니다. Poetry 가상환경 대신 코랩의 시스템 환경에 직접 `pip`를 사용해 신속하게 설치합니다.

In [ ]:
# 파이프라인 수행에 필요한 라이브러리 패키지 일괄 설치
!pip install pandas numpy xgboost scikit-learn python-dotenv pymongo requests publicdatareader finance-datareader pykrx torch transformers accelerate sentence-transformers sentencepiece tiktoken protobuf

### 3단계. 환경 변수(.env) 설정 및 Secrets 연동

보안이 필요한 API 키와 DB 접속 URI는 코드에 노출하지 않고 **Colab Secrets**를 사용합니다.

#### 🛠️ 설정 방법:
1. 코랩 좌측 메뉴에서 **열쇠 모양 아이콘(비밀번호/Secrets)**을 클릭합니다.
2. 다음 이름(Name)으로 비밀 값(Value)을 각각 추가하고 **Notebook 액세스 허용** 토글을 켜줍니다:
   * `MONGODB_URI` (MongoDB 접속 주소 - *중요: MongoDB Atlas IP Access List에 0.0.0.0/0 허용이 되어 있어야 코랩에서 접근이 가능합니다*)
   * `HF_TOKEN` (HuggingFace Access Token)
   * `DART_API_KEY` (OpenDART API Key)
   * `ECOS_API_KEY` (한국은행 ECOS API Key)
   * `NCP_API_KEY_ID` (Naver Cloud Platform Cloud API ID)
   * `NCP_API_KEY` (Naver Cloud Platform Cloud API Secret Key)
3. 아래 셀을 실행하면 Secrets에서 키 값을 읽어와 `backend/.env` 파일을 임시로 생성해 줍니다. 기존 로컬 파이썬 스크립트들이 소스코드 변경 없이 정상 작동하게 됩니다.

In [ ]:
import os
from google.colab import userdata

keys = [
    "MONGODB_URI",
    "HF_TOKEN",
    "DART_API_KEY",
    "ECOS_API_KEY",
    "NCP_API_KEY_ID",
    "NCP_API_KEY"
]

env_content = ""
print("🔑 Colab Secrets 로드 및 backend/.env 생성 시작...")
for key in keys:
    try:
        val = userdata.get(key)
        if val:
            env_content += f"{key}={val}\n"
            print(f"  ✅ {key} 로드 완료")
        else:
            print(f"  ⚠️ {key} 값이 비어있습니다.")
    except Exception as e:
        print(f"  ❌ {key} 로드 실패 (Secrets 설정을 확인하세요): {e}")

# backend 디렉터리 존재 확인 후 .env 파일 쓰기
os.makedirs("backend", exist_ok=True)
with open("backend/.env", "w", encoding="utf-8") as f:
    f.write(env_content)

print("\n🎉 backend/.env 파일이 성공적으로 작성되었습니다!")

### 4단계. 통합 파이프라인 실행

이제 모든 준비가 끝났습니다. 아래 셀을 실행하면 다음 단계가 자동으로 순차 수행됩니다:
1. **주가 데이터 수집** (`collect_price.py`)
2. **주가 라벨 생성** (`generate_labels.py`)
3. **업종 매핑** (`add_sector.py`)
4. **최근 뉴스 수집** (`collect_news.py`)
5. **뉴스 텍스트 DL 분석 및 피처 분류** (`process_news_features.py` - *GPU 연산으로 진행*)
6. **최종 학습/추론 통합 피처 데이터셋 병합** (`join_features.py`)
7. **XGBoost 피처 중요도 및 성능 비교 실험** (`compare_features.py`)

수행된 모든 로그는 `data/pipeline_run_latest.log` 파일에 저장되며 콘솔에도 실시간 출력됩니다.

In [ ]:
# 통합 파이프라인 구동
!python backend/scripts/run_pipeline.py

### 5단계. 개별 스크립트 실행 (필요 시)

전체 파이프라인 구동 외에, 개별 작업만 별도로 수행하고 싶을 경우 아래 주석을 해제하여 사용하세요.

In [ ]:
# 예: 뉴스 데이터만 단독으로 다시 수집
# !python backend/scripts/collect_news.py

# 예: 수집된 뉴스를 가져와 SBERT 중복제거 및 카테고리/감성 판정만 다시 실행
# !python backend/scripts/process_news_features.py

# 예: 통합된 피처들을 기준으로 XGBoost 학습 실험만 다시 실행
# !python backend/scripts/compare_features.py

# 예: 학습된 모델 기반으로 최종 daily_risk_score를 예측하여 DB에 저장
# !python backend/scripts/save_risk_scores.py